# InfraHeal AI — Autonomous Incident Diagnosis & Resolution Agent

**Team:** team-790 (bharathram.naiktv)  
**Track:** Track 1 – Agents (AGENTS_026)  
**Stack:** AMD ROCm + vLLM + Qwen2.5-7B-Instruct  
**TCS & AMD AI Hackathon 2026**

---
### Cell 0: Setup & Imports

In [ ]:
import sys, os, json, time, logging
from pathlib import Path

REPO_DIR = '/workspace/shared/infraheal-ai'
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)-25s | %(levelname)-7s | %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('infraheal.notebook')

# Import core components
from config import VLLM_BASE_URL, VLLM_API_KEY, MODEL_NAME, detect_model
from data_generator import generate_all_data, create_incident_scenarios
from anomaly_detector import AnomalyDetector
from rag.knowledge_base import KnowledgeBase
from agents.orchestrator import InfraHealOrchestrator
from gpu_tracker import GPUMonitor

print(f'Working dir: {os.getcwd()}')
print('All imports successful')

---
### Cell 1: vLLM Connection Test

Connect to the AMD GPU vLLM server and verify model availability.

In [ ]:
from config import VLLM_BASE_URL, VLLM_API_KEY, MODEL_NAME, detect_model
from openai import OpenAI
import time

print(f'Connecting to vLLM at {VLLM_BASE_URL}...')
client = OpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY,
                timeout=60.0, max_retries=2)

models = client.models.list()
available = [m.id for m in models.data]
print(f'Available models: {available}')

model = detect_model(client)
print(f'Selected model: {model}')

start = time.time()
response = client.chat.completions.create(
    model=model,
    messages=[{'role': 'user', 'content': 'Say "InfraHeal AI online" in 3 words.'}],
    max_tokens=20,
    temperature=0.1,
)
elapsed = time.time() - start
reply = response.choices[0].message.content
tokens = response.usage.total_tokens if response.usage else 'N/A'
print(f'Inference OK in {elapsed:.2f}s | Tokens: {tokens} | Reply: {reply}')
print('vLLM READY')

---
### Cell 2: Generate Synthetic Data

Generate realistic infrastructure logs, metrics, runbooks, and past incidents.

In [ ]:
print('Generating synthetic infrastructure data...')
data = generate_all_data(save_to_disk=True)
print(f'Logs: {len(data["logs"]):,}')
print(f'Metrics: {len(data["metrics"]):,}')
print(f'Runbooks: {len(data["runbooks"])}')
print(f'Past Incidents: {len(data["past_incidents"])}')

scenarios = create_incident_scenarios()
for sc in scenarios:
    print(f'  [{sc["id"]}] {sc["name"]} (sev={sc["expected_severity"]})')
print('DATA READY')

---
### Cell 3: Initialize Knowledge Base & Anomaly Detector

Set up the BM25 RAG pipeline and anomaly detection engine.

In [ ]:
kb = KnowledgeBase(runbooks=data['runbooks'], past_incidents=data['past_incidents'])
detector = AnomalyDetector()

ctx = kb.get_context('database connection pool exhaustion', category='database')
print(f'KB context: {len(ctx)} chars')
print(ctx[:300] + '...')

sc = scenarios[0]
anomalies = detector.detect_all(logs=sc['logs'], metrics=sc['metrics'])
print(f'\n{sc["name"]}: {len(anomalies)} anomalies detected')
for a in anomalies[:3]:
    print(f'  [{a["severity"]}] {a["type"]}: {a["description"][:100]}')
print('SYSTEMS INITIALIZED')

---
### Cell 4: Run Full Agent Pipeline

Run Triage → RCA → Remediation → Reporting pipeline on a real incident scenario.

In [ ]:
orchestrator = InfraHealOrchestrator(client=client, model_name=model, knowledge_base=kb)

scenario = scenarios[1]
anomalies = detector.detect_all(logs=scenario['logs'], metrics=scenario['metrics'])

print(f'Processing: {scenario["name"]}')
print(f'Anomalies detected: {len(anomalies)}')
print('Running 4-agent pipeline...')

start = time.time()
result = orchestrator.process_incident(
    anomalies=anomalies,
    logs=scenario['logs'],
    metrics=scenario['metrics'],
)
elapsed = time.time() - start

print(f'\nPipeline complete in {elapsed:.2f}s')
print(json.dumps(result.get('pipeline_metrics', {}), indent=2))

---
### Cell 5: Review Agent Outputs

Examine structured results from each agent.

In [ ]:
triage = result.get('triage_result', {})
rca_res = result.get('rca_result', {})
remediation = result.get('remediation_result', {})
report = result.get('report', {})

print('=' * 60)
print('TRIAGE AGENT OUTPUT')
print('=' * 60)
print(f'Severity: {triage.get("severity")} ({triage.get("severity_label")})')
print(f'Category: {triage.get("category")}')
print(f'Impact: {triage.get("impact_assessment")}')
print(f'Confidence: {triage.get("confidence", 0):.2f}')
print(f'Services: {triage.get("affected_services", [])}')
print()

print('=' * 60)
print('RCA AGENT OUTPUT')
print('=' * 60)
print(f'Root Cause: {rca_res.get("root_cause")}')
print(f'Confidence: {rca_res.get("confidence_score", 0):.2f}')
print(f'Blast Radius: {rca_res.get("blast_radius")}')
print(f'Evidence:')
for e in rca_res.get('evidence_chain', [])[:3]:
    print(f'  - {e}')
print()

print('=' * 60)
print('REMEDIATION AGENT OUTPUT')
print('=' * 60)
print(f'Execution order: {remediation.get("execution_order")}')
print(f'ETA: {remediation.get("estimated_resolution_time")}')
print(f'Confidence: {remediation.get("confidence", 0):.2f}')
for act in remediation.get('recommended_actions', []):
    print(f'  Step {act["step"]}: {act["tool_name"]} ({act["risk_level"]} risk)')
    print(f'    {act["rationale"][:100]}')
print()

print('=' * 60)
print('INCIDENT REPORT')
print('=' * 60)
print(f'ID: {report.get("incident_id")}')
print(f'Title: {report.get("title")}')
print(f'Status: {report.get("status")}')
print(f'Summary: {report.get("executive_summary")[:200]}...')

---
### Cell 6: Reasoning Chain Visualization

Show the agent-by-agent reasoning chain.

In [ ]:
print('AGENT REASONING CHAIN')
print('=' * 60)
for step in result.get('reasoning_chain', []):
    print(f'--- {step["agent"]} ---')
    print(f'  {step["reasoning"][:200]}')
    print(f'  Latency: {step.get("latency_ms", 0)}ms\n')

---
### Cell 7: Interactive 2D Visualizations

Plotly-powered visualizations — time-series dashboard, correlation heatmap, anomaly timeline, log distribution.

In [ ]:
import json, plotly.io as pio
pio.renderers.default = 'notebook'

from visualizer_2d import (
    time_series_dashboard,
    correlation_heatmap,
    anomaly_timeline,
    log_level_distribution,
    host_radar,
)
from anomaly_detector import IncidentCorrelator

with open('sample_data/metrics.json') as f:
    full_metrics = json.load(f)
with open('sample_data/logs.json') as f:
    full_logs = json.load(f)

detected = detector.detect_all(logs=full_logs[:2000], metrics=full_metrics[:500])
incidents = IncidentCorrelator().correlate(detected)

fig1 = time_series_dashboard(full_metrics[:500], anomalies=detected[:10],
                              title='Metric Time-Series Dashboard')
fig2 = correlation_heatmap(full_metrics[:500], title='Metric Correlation Heatmap')
fig3 = anomaly_timeline(incidents, title='Anomaly Timeline by Host') if incidents else \
       host_radar(full_metrics[:500], title='Host Comparison')
fig4 = log_level_distribution(full_logs[:2000], title='Log Level Distribution')

print(f'Generated {4} plots — displaying inline below!')
fig1.show()
fig2.show()
fig3.show()
fig4.show()

---
### Cell 8: Performance Benchmarks (for Slide 4)

Capture metrics required by the hackathon submission: tokens, latency, GPU memory.

In [ ]:
pm = result.get('pipeline_metrics', {})
agent_m = pm.get('agent_metrics', {})
totals = agent_m.get('totals', {})

print('PERFORMANCE METRICS (for Slide 4)')
print('=' * 60)
print(f'Total pipeline time: {pm.get("total_time_seconds", 0):.2f}s')
print(f'Breakdown:')
print(f'  Triage: {pm.get("triage_latency", 0):.2f}s')
print(f'  RAG lookup: (included in RCA)')
print(f'  RCA: {pm.get("rca_latency", 0):.2f}s')
print(f'  Remediation: {pm.get("remediation_latency", 0):.2f}s')
print(f'  Reporting: {pm.get("report_latency", 0):.2f}s')
print(f'Total tokens: {totals.get("total_tokens", 0):,}')
print(f'LLM calls: {totals.get("total_calls", 0)}')
print(f'Errors: {totals.get("total_errors", 0)}')
print(f'GPU memory: {pm.get("gpu_memory_mb", 0):.0f} MB')
print(f'GPU peak: {pm.get("gpu_peak_memory_mb", 0):.0f} MB')
print(f'\nPer-agent:')
for agent_key, agent_name in [('triage', 'Triage'), ('rca', 'RCA'),
                                ('remediation', 'Remediation'), ('reporting', 'Reporting')]:
    a = agent_m.get('agents', {}).get(agent_key, {})
    print(f'  {agent_name}: {a.get("total_tokens", 0)} tokens, '
          f'{a.get("avg_latency", 0)*1000:.0f}ms avg')

---
### Cell 9: Launch Interactive Dashboard

Start the Gradio dashboard with live anomaly detection and agent visualization.

In [ ]:
from jupyter_helper import launch_dashboard

demo = launch_dashboard(share=True, port=7860)
print('Dashboard running at the share URL above.')

---
### Cell 10: Stress Test — Run All Scenarios

Process every incident scenario end-to-end to collect comprehensive benchmarks.

In [ ]:
print('Running all scenarios for benchmark...')
all_results = []
for sc in scenarios:
    anomalies = detector.detect_all(logs=sc['logs'], metrics=sc['metrics'])
    res = orchestrator.process_incident(
        anomalies=anomalies,
        logs=sc['logs'],
        metrics=sc['metrics'],
    )
    all_results.append(res)
    pm = res.get('pipeline_metrics', {})
    print(f'[{sc["id"]}] {sc["name"]}: {pm.get("total_time_seconds", 0):.2f}s, '
          f'{pm.get("agent_metrics", {}).get("totals", {}).get("total_tokens", 0)} tokens')

total_time = sum(r.get('pipeline_metrics', {}).get('total_time_seconds', 0) for r in all_results)
total_tokens = sum(r.get('pipeline_metrics', {}).get('agent_metrics', {}).get('totals', {}).get('total_tokens', 0) for r in all_results)
print(f'\nTOTAL: {len(scenarios)} scenarios in {total_time:.2f}s, {total_tokens:,} tokens')

---
### Cell 11: Save Results for Presentation

Export pipeline results to JSON for inclusion in the submission PPT.

In [ ]:
output = {
    'scenarios': [
        {
            'id': sc['id'],
            'name': sc['name'],
            'anomalies': len(detector.detect_all(logs=sc['logs'], metrics=sc['metrics'])),
        }
        for sc in scenarios
    ],
    'results': [
        {
            'id': r.get('pipeline_id'),
            'severity': r.get('triage_result', {}).get('severity'),
            'root_cause': r.get('rca_result', {}).get('root_cause'),
            'metrics': r.get('pipeline_metrics'),
        }
        for r in all_results
    ],
    'total_tokens': total_tokens,
    'total_time': total_time,
    'average_time_per_incident': total_time / max(len(scenarios), 1),
}

with open('benchmark_results.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)

print('Benchmark results saved to benchmark_results.json')
print(json.dumps({'total_tokens': total_tokens, 'total_time_s': round(total_time, 2)}, indent=2))